# 10 — Four-pipeline comparison

Thin orchestration only — all logic lives in `src/reviewscope_ml/`.

Runs the four end-to-end candidates on the benchmark sample and renders the
comparison report:

| Variant | Description |
|---|---|
| `bertopic` | BERTopic off-the-shelf (MiniLM, its own UMAP+HDBSCAN; seeded) |
| `custom_hdbscan` | mpnet → UMAP(10d) → HDBSCAN (notebooks 04–06 decisions) |
| `flat_agglomerative` | mpnet → UMAP(10d) → agglomerative (ward) |
| `two_stage` | fine HDBSCAN micro-clusters → agglomerative macro merge |

**Decision protocol** (also embedded in the report): metrics shortlist 2–3
finalists; a human picks the winner from the qualitative inspection and the
intruder test. Metrics never declare the winner alone.

Runs top-to-bottom on CPU at 1k in a few minutes (embeddings are cached after
the first run). For the full 5k/50k GPU runs see `docs/methodology.md`.

In [1]:
import sys
sys.path.insert(0, '..')  # notebooks/utils shims; package itself is pip-installed

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(message)s')

from reviewscope_ml import load_config

# CPU smoke configuration — keep this runnable on any laptop.
# GPU: use reviewscope_ml.runtime.claim_gpu() and pass device/gpu_id (see docs).
cfg = load_config(sample_size=1_000, device='cpu')
cfg.ensure_dirs()
print(cfg)

PipelineConfig(
  sample_size  = 1,000
  data_file    = sample_hotels_1k.jsonl
  device       = cpu
  gpu_id       = —
  batch_size   = 64
  cpu_threads  = 4
  seed         = 42
  cache_dir    = /home/4dmarten/repo/ReviewScope/data/cache
)


In [2]:
# Benchmark sample (built from the raw Yelp dump if missing; idempotent)
from reviewscope_ml.data import build_benchmark_sample, subset_sample

five_k = build_benchmark_sample(cfg.project_root, 5_000)
subset_sample(five_k, cfg.data_path, cfg.sample_size)
print('benchmark:', cfg.data_path)

2026-06-11 15:00:34,031 reviewscope.data benchmark sample already exists: /home/4dmarten/repo/ReviewScope/data/cache/sample_hotels_5k.jsonl


benchmark: /home/4dmarten/repo/ReviewScope/data/cache/sample_hotels_1k.jsonl


In [3]:
# Run all four variants + multi-seed stability repeats, write the report.
# label_clusters=True uses Ollama when reachable and falls back to term
# labels (recorded as label_source='terms_fallback') when not.
from reviewscope_ml.eval.report import run_comparison

report_path = run_comparison(cfg, seeds=(42, 43, 44), label_clusters=True)
report_path

2026-06-11 15:00:34,109 reviewscope.report === variant bertopic (seed 42) ===


2026-06-11 15:00:34,110 reviewscope.pipeline run bertopic__1000__s42 already complete — loading artifacts


2026-06-11 15:00:34,117 reviewscope.report --- stability repeat: bertopic seed 43 ---


2026-06-11 15:00:36,318 reviewscope.embed released embedding model sentence-transformers/all-MiniLM-L6-v2


2026-06-11 15:00:36,320 reviewscope.report --- stability repeat: bertopic seed 44 ---


...

  [loaded] s44_all-mpnet-base-v2__no_inst__nc10__nn15__md000__cosine__1k.npy  shape=(1000, 10)  dtype=float32
  [loaded] agglomerative__k15__linkageward__s44_all-mpnet-base-v2__no_inst__nc10__nn15__1k.npy  shape=(1000,)  dtype=int32
  [loaded] all-mpnet-base-v2__no_inst__1k.npy  shape=(1000, 768)  dtype=float32
  [loaded] s43_all-mpnet-base-v2__no_inst__nc10__nn15__md000__cosine__1k.npy  shape=(1000, 10)  dtype=float32
  [loaded] two_stage__linkageward__mmcs5__mms3__nmacauto__s43_all-mpnet-base-v2__no_inst__nc10__nn15__1k.npy  shape=(1000,)  dtype=int32
  [loaded] micro_two_stage__linkageward__mmcs5__mms3__nmacauto__s43_all-mpnet-base-v2__no_inst__nc10__nn15__1k.npy  shape=(1000,)  dtype=int32
  [loaded] all-mpnet-base-v2__no_inst__1k.npy  shape=(1000, 768)  dtype=float32


2026-06-11 15:00:37,258 reviewscope.embed released embedding model sentence-transformers/all-mpnet-base-v2


  [loaded] s44_all-mpnet-base-v2__no_inst__nc10__nn15__md000__cosine__1k.npy  shape=(1000, 10)  dtype=float32
  [loaded] two_stage__linkageward__mmcs5__mms3__nmacauto__s44_all-mpnet-base-v2__no_inst__nc10__nn15__1k.npy  shape=(1000,)  dtype=int32
  [loaded] micro_two_stage__linkageward__mmcs5__mms3__nmacauto__s44_all-mpnet-base-v2__no_inst__nc10__nn15__1k.npy  shape=(1000,)  dtype=int32


2026-06-11 15:00:38,101 reviewscope.report comparison report -> /home/4dmarten/repo/ReviewScope/data/runs/comparison_1000/report.md


PosixPath('/home/4dmarten/repo/ReviewScope/data/runs/comparison_1000/report.md')

In [4]:
# Render the report inline (same file a teammate reads on disk)
from IPython.display import Markdown

Markdown(report_path.read_text())

# Pipeline comparison — 1,000 hotel reviews

Benchmark: `sample_hotels_1k.jsonl` · device: cpu · seeds: [42, 43, 44]

## How to read this report

This comparison follows a two-step protocol:

1. **Metrics shortlist** (this file, automated): three-tier metrics, stability, and structural failure flags narrow the field to 3 finalists. Metrics are cheap proxies — they rank, they never declare a winner.
2. **Human inspection picks the winner**: read the qualitative inspection and take the intruder test below for each finalist; then confirm (or veto) in the HITL review app, which records the decision. A pipeline whose clusters you cannot name in one phrase loses, whatever its scores.

## Tier overview

| Metric | What it measures | Known bias |
|---|---|---|
| Silhouette (excl. noise) | geometric separation in reduced space | rewards discarding hard points as noise; rewards sentiment polarity |
| Silhouette (incl. noise) | same, noise as own pseudo-cluster | fairer to partitioners; punishes honest noise labeling |
| C_v coherence | top-term co-occurrence in raw text | unreliable on very short texts; insensitive to boundary quality |
| Rating entropy | star-rating mix within clusters (high = thematic) | needs a rating column; hotel-specific calibration |
| ARI stability | agreement of clusterings across seeds | says nothing about quality, only repeatability |

## Results

| Variant | clusters | noise | sil (excl) | sil (incl) | C_v | entropy | max share | ARI mean | ARI min | runtime s |
|---|---|---|---|---|---|---|---|---|---|---|
| bertopic | 21 | 0.356 | 0.631 | 0.217 | 0.467 | 0.727 | 0.090 | 0.568 | 0.502 | 95.620 |
| custom_hdbscan | 7 | 0.010 | 0.475 | 0.340 | 0.479 | 0.895 | 0.630 | 0.543 | 0.320 | 199.890 |
| flat_agglomerative | 15 | 0.000 | 0.456 | 0.456 | 0.463 | 0.768 | 0.132 | 0.702 | 0.634 | 0.720 |
| two_stage | 7 | 0.263 | 0.483 | 0.238 | 0.472 | 0.805 | 0.294 | 0.446 | 0.379 | 0.490 |

### Per-stage cost

| Variant | stage | wall s | RSS peak MB | VRAM peak MB |
|---|---|---|---|---|
| bertopic | embed | 46.57 | 1495.0 | 0.0 |
| bertopic | reduce_cluster | 42.81 | 1495.0 | 0.0 |
| bertopic | viz_coords | 6.04 | 1495.0 | 0.0 |
| bertopic | represent | 0.2 | 1495.0 | 0.0 |
| bertopic | label | 0.0 | 1495.0 | 0.0 |
| bertopic | evaluate | 2.65 | 1495.0 | 0.0 |
| custom_hdbscan | embed | 189.02 | 3711.0 | 0.0 |
| custom_hdbscan | reduce | 3.75 | 3711.0 | 0.0 |
| custom_hdbscan | cluster | 0.03 | 3711.0 | 0.0 |
| custom_hdbscan | viz_coords | 6.85 | 3711.0 | 0.0 |
| custom_hdbscan | represent | 0.24 | 3711.0 | 0.0 |
| custom_hdbscan | label | 0.0 | 3711.0 | 0.0 |
| custom_hdbscan | evaluate | 1.77 | 3711.0 | 0.0 |
| flat_agglomerative | embed | 0.46 | 3711.0 | 0.0 |
| flat_agglomerative | reduce | 0.0 | 3711.0 | 0.0 |
| flat_agglomerative | cluster | 0.02 | 3711.0 | 0.0 |
| flat_agglomerative | viz_coords | 0.0 | 3711.0 | 0.0 |
| flat_agglomerative | represent | 0.24 | 3711.0 | 0.0 |
| flat_agglomerative | label | 0.0 | 3711.0 | 0.0 |
| flat_agglomerative | evaluate | 1.86 | 3711.0 | 0.0 |
| two_stage | embed | 0.3 | 1211.3 | 0.0 |
| two_stage | reduce | 0.0 | 1211.3 | 0.0 |
| two_stage | cluster | 0.0 | 1211.3 | 0.0 |
| two_stage | viz_coords | 0.0 | 1211.3 | 0.0 |
| two_stage | represent | 0.19 | 1211.3 | 0.0 |
| two_stage | label | 0.0 | 1211.3 | 0.0 |
| two_stage | evaluate | 1.48 | 1265.2 | 0.0 |

### Failure flags

- **bertopic**: no structural flags
- **custom_hdbscan**:
  - giant cluster: one cluster holds 63% of all documents (blob + crumbs pattern)
- **flat_agglomerative**:
  - near-duplicate clusters 1 and 4: 70% top-term overlap — candidates for merging
  - near-duplicate clusters 1 and 10: 60% top-term overlap — candidates for merging
  - near-duplicate clusters 4 and 5: 60% top-term overlap — candidates for merging
  - near-duplicate clusters 4 and 10: 80% top-term overlap — candidates for merging
  - near-duplicate clusters 5 and 10: 60% top-term overlap — candidates for merging
- **two_stage**:
  - near-duplicate clusters 0 and 1: 60% top-term overlap — candidates for merging

![metrics](charts_metrics.png)

## Step 1 result — shortlisted finalists

Mean rank across silhouette, silhouette_incl_noise, coherence_cv, rating_entropy shortlists: **custom_hdbscan**, **two_stage**, **bertopic**

Noise-fairness reminder: HDBSCAN-family variants discard noise, which inflates the classic silhouette — compare the *incl.-noise* column and the noise fraction before trusting Tier-1 differences.

## Step 2 — human inspection of the finalists

## Qualitative inspection — `custom_hdbscan__1000__s42`

_Samples are random cluster members (not centroid-nearest). Ask per cluster: can you name this in one phrase, and do the samples fit that phrase?_

**Noise:** 10 documents (1%) unassigned.

### Cluster 0 — “oysters / good / food” (78 docs · avg 3.81★, label: terms_fallback)
**Top terms:** oysters, good, food, shrimp, place, service, ordered, great

- Everything about this place was wonderful. The waitress was lovely and I loved the architecture. The food was amazing! I had a salad and green beans which were probably one of the greatest things I've ever had before. I definitely recommend coming here. Everyone is so nice! They …
- Food was a 3.5 or so, but our server Kiki pushed our rating up to 4. Oysters on half shell were good (but so damn big!). BBQ oysters were really tasty. Turtle soup - solid and spicy. BBQ shrimp - wife loved it, I thought just ok.
- It's not a bad place to go, but I think it's a bit overpriced. The views are great and probably the best part about the experience. The food is decent, but I would rather eat downstairs at Bongos on the beach and I'm pretty sure the food comes from the same kitchen. Not a huge se…
- Great service, great oysters. All the doors/windows open air to Bourbon St. Nice place for lunch.
- CAJUN Bloody Mary's! I have to write this because I searched yelp up and down for a true bloody mary on Bourbon St. Finally I came across this place by accident, with pickled green beans. LOVE LOVE LOVE.

### Cluster 1 — “nashville / hotel / opryland” (29 docs · avg 3.72★, label: terms_fallback)
**Top terms:** nashville, hotel, opryland, like, resort, downtown, really, room

- My friends and I came here for two nights from Chicago. Everyone in Nashville is just super nice! It's kind of weird! Haha. The rooms were pretty cheap compared to other hotels and they were very clean! I loved the fact that they had a free shuttle that would take you to downtown…
- The only knock on this place is that they didn't have all the pools open during Memorial Day weekend. Other than that the hotel provides a very friendly environment that's good for adults and families as well plenty of places to eat inside the resort and the atrium was amazing an…
- We booked a garden view room with a balcony for 2 nights. The room itself was ok, but it was the hotel grounds that made the stay. The hotel is obviously built to keep you within its property. The inside is like a maze, but a beautiful one. Huge gardens, an indoor river, boutique…
- Ok, my initial disappointment has been somewhat eliviated, but the Opryland Resort is still lacking a world-class resort feel. Perhaps it's the purpose and theme of the hotel. There are some really nice amenities, especially the shopping. The restaurants are somewhat disappointin…
- I preface my three stars with this - I really did enjoy this hotel, and I would come back again, but there were some things that stood out a took away from a 4 star rating. Let's begin with the positives: 1. The decor/Character. It's awesome - fresh, youthful yet historic, lively…

### Cluster 2 — “jw / hotel / indianapolis” (18 docs · avg 4.33★, label: terms_fallback)
**Top terms:** jw, hotel, indianapolis, room, space, staff, meeting, marriott

- Though top notch service is demanded at a JW, I was thoroughly impressed with the property, service, and staff. I only had two complaints. One was the noise level as there was an outdoor concert until 11pm. The other was the below average bartender at velocity during lunch. Nothi…
- Great external look, amazing five-star staff, and outstanding meeting space and capabilitiesmake this place very worthy of five-stars, but issues with the functionality of meeting space accommodations render this hotel four stars. Riding to the hotel in an Uber, i didn't know wha…
- Honestly probably one of the top hotels at which I've ever stayed. I stayed here for a conference and received a King room on a high floor with a view of the city (the park is the other view option). I LOVE the city view which overlooks downtown Indy and I even got to watch the s…
- Great hotel, the art creates an amazing atmosphere. Up to date, clean, modern rooms that look great. Plenty of shower room, and the overhead shower heads should be mandatory in every other hotel! Who knew this little gem existed in Indianapolis! Staff was nice, check in was easy.…
- The JW Marriott in downtown Indianapolis is absolutely beautiful with some of the best views of the city. The hotel has anything you could ask for from restaurants to the High Velocity Sports Bar to Starbucks. A few friends and I stayed here for a Bachelorette Party and its locat…

### Cluster 3 — “casino / reno / peppermill” (98 docs · avg 4.1★, label: terms_fallback)
**Top terms:** casino, reno, peppermill, room, place, best, vegas, rooms

- The Peppermill is beautiful, clean and easy to get around for such a large hotel. It's the only hotel we will stay at in Reno from now on.
- So disappointed in your company's customer service!!! I am turning 30 in July and have family coming to celebrate it with me, we all bought the Costco package deals which is not much less then your daily rates and yesterday my sister was on the phone with customer service who was…
- The Peppermill is always making upgrades to their hotel to make it look better. But if you're a non smoker, good luck on making it through without getting second hand smoke cancer. They used to have these portable air purifiers all over the casino but they have steadily disappear…
- Oh, Peppermill...you used to be my home away from home but lately, I just need to say a few things... I started staying with you back in 2008. A coworker recommended your lavish Tuscany suites and what a sweet deal it was. I enjoyed many fun times staying with you... from weekend…
- Underwhelming hotel, overwhelming experience, overpriced everything, humid/smelly atmosphere, and loud. I don't understand why anyone likes this? Yes, they have fake waterfalls and real plants inside, but it's so crowded, it's HUGE (literally could take 20 mins to get to your roo…

### Cluster 4 — “hotel / orleans / great” (107 docs · avg 4.17★, label: terms_fallback)
**Top terms:** hotel, orleans, great, quarter, french, room, bar, new

- Absolutely one of the best experiences for myself and my family. This review is waaaaay overdue. We arrived to Terrell House for a wedding. The place is spectacular. Like stepping into the 1800s, except with all the modern accomodations (AC, etc). Linda, the owner is a sweetheart…
- Ok. Room was nice but nothing special. They are in the middle of renovations so I'm sure it will be nicer soon. Staff was very nice. I prefer to be further east towards Frenchman St but this was a nice place to stay.
- We did the "park and cruise" package as we needed a place to stay before our cruise left NOLA. Dude, it's $150 for 1 night and a week's parking in the garage. You'd pay that much for the parking alone! Totally worth it. So even if the room weren't essentially free because of the …
- Ace hotel is one of my favorite places in the city. It has it all. It reminds me of Miami, I'm not sure why, but it does. The vibe is a bit more sheek and upscale. I know its New Orleans and anything goes but please don't show up in shorts and flip flops. Prices are high but you …
- Was in the area over the weekend for a family reunion and most of the family was staying in this hotel. The location of the hotel was very convenient, about 15 minutes from downtown St. Louis and 15 minutes from the zoo which my daughter loved. Just wish it wasn't so hot while we…

### Cluster 5 — “hotel / philly / city” (30 docs · avg 4.13★, label: terms_fallback)
**Top terms:** hotel, philly, city, room, philadelphia, stay, great, night

- On a recent work trip I picked Loews among the other similarly priced options on hotels.com (probably why I did not receive a welcome goodie bag) b/c I prefer more modern decor and it was close to my meeting location. My room was quiet and clean, but if I compare it to somewhere …
- This was an amazing hotel from the service to the room. Everything was impeccable taken care of. The staff were very friendly and helpful. Room was spacious, well laid out and very clean. It is centrally located that allows you to get to places very quickly by foot. If you are in…
- Excellent location!! This hotel sits kitty-corner from Independence Hall entrance, right across the street from Liberty Bell and a stones-throw from many other historical significant sites in Philadelphia. It is also an easy stroll to Penn's Landing, South Street and two blocks d…
- Just to let you know I'm from Jersey but LOVE to go out to Phila. a lot! So, I was in the city to see HIM and I booked a hotel via Hotwire and they gave me a pretty good deal but the experienced sucked my window was facing another building just garbage when you are looking to see…
- Overall, I enjoyed my last couple stays at this location. The rooms are clean, the bathrooms are clean and the wireless internet is fast. There is also HBO and clear and direct TV stations. The hotel is next to a WAWA which offers 24 hour gas and made to order sandwiches. Most of…

### Cluster 6 — “room / hotel / stay” (630 docs · avg 3.4★, label: terms_fallback)
**Top terms:** room, hotel, stay, great, staff, nice, clean, rooms

- This hotel has wonderful views of the airport, and it has the only revolving rooftop restaurant. It's connected to the airport that has several restaurants and shops. From the revolving rooftop restaurant, I could see the runways, and the control tower.
- Couldn't have asked for a better location but horrible maid service and not worth the price. Horrible communication between the staff. The room was nice but didn't include a mini fridge or blow drier. The wi-fi was also a problem. It stopped working and when I asked the front des…
- As the man at the front desk just told me, check-in "begins at 3, that doesn't mean a room will be available." But we can check our bags for a fee and have a drink at the bar (not complimentary) while we wait. Right now the lobby is crammed full of people waiting for them to get …
- After speaking with the manager the next morning, she sincerely apologized for the lack of quality customer service we had received the night before. She gave us a bottle of wine and said the next time we are in the area, to ask for her and she would make sure our stay was a plea…
- The SPA is my greatest new find (plan on spending an entire day here), they have many areas just to lounge, incredible services, great snacks to keep you nourished.. We stayed in the Tuscany Tower on a non-smoking floor and the room was fabulous.


## Intruder test — `custom_hdbscan__1000__s42`

_Each block shows 4 documents from one cluster and 1 from another, shuffled. If the intruder is not obvious, the boundary between the two clusters is not meaningful. Answer key at the bottom._

### Block 0 — cluster “oysters / good / food”
1. Huge oysters when I was there. Great service! Had a muffuletta (half). It was all very tasty!
2. Solid, chill dive bar. Bartender super nice and funny as hell. A local recommended the Hurricane at this bar and he did not steer us wrong. Holy sh*t! Yes, it is really tasty and good. And STRONG! Compared to PO'B, this Hurricane is better. It's the mix...I believe they use real …
3. Everything about this place was wonderful. The waitress was lovely and I loved the architecture. The food was amazing! I had a salad and green beans which were probably one of the greatest things I've ever had before. I definitely recommend coming here. Everyone is so nice! They …
4. Great service, great oysters. All the doors/windows open air to Bourbon St. Nice place for lunch.
5. The location is great for those staying and touring the French Quarter. Use the valet...parking in the Quarter is a headache.

### Block 1 — cluster “nashville / hotel / opryland”
1. The ultimate place to stay in Nashville. I have been fortuate to stay there twice. Seems a little overwhelming at first with its sheer size you could get lost, but the staff is really helpful and they have maps all around. The restuarants run the gambit from moderately priced to …
2. I visited the Gaylord Opryland hotel/resort/conglomerate a few weeks ago when I was visiting Nashville with my boyfriend. We stopped by to see the hotel, but didn't stay there as guests. This place wasn't really near any downtown things, but we rented a car so it was no big deal …
3. The JW Marriott in downtown Indianapolis is absolutely beautiful with some of the best views of the city. The hotel has anything you could ask for from restaurants to the High Velocity Sports Bar to Starbucks. A few friends and I stayed here for a Bachelorette Party and its locat…
4. Hotel with good amenities including an in-house gym. The front desk staff was extremely friendly and helpful, especially James. Free coffee in the lobby is a plus, but there is no free breakfast to kick off the day. On the other hand, there is a small coffee stand (with some food…
5. So, I have never actually stayed at the Opryland hotel because well I live here and who cares. However, I am rating the lobby because even a non corporate, simple, no frills gal like me must give pause to a hotel that has a river with boat tours in it's lobby. All the plants are …

### Block 2 — cluster “jw / hotel / indianapolis”
1. The JW Marriott in downtown Indianapolis is absolutely beautiful with some of the best views of the city. The hotel has anything you could ask for from restaurants to the High Velocity Sports Bar to Starbucks. A few friends and I stayed here for a Bachelorette Party and its locat…
2. I cannot say enough about the accommodations at the JW. I was working a trade show in the Grand Ballroom today and needed to pump as I am a new mom. Nathan was so quick to help as I was searching for a location that I could use. He opened up a boardroom for me to use and locked i…
3. Honestly probably one of the top hotels at which I've ever stayed. I stayed here for a conference and received a King room on a high floor with a view of the city (the park is the other view option). I LOVE the city view which overlooks downtown Indy and I even got to watch the s…
4. New York in Indianapolis. Beautiful room with the nicest of linens, soft towels, high quality soaps and exceptional service. Walking distance to Lucas Oil Stadium and downtown if you like a short hike. Parking cost was also New York caliber. We grabbed a quick bite in the bar/res…
5. Lovely boutique hotel! Room was pleasant - a little small but certainly enough space for 2 good friends. Very convenient to almost anywhere in the city. Good relationships with neighboring businesses such as restaurants, gyms and parking facilities. Had a reunion weekend with a c…

### Block 3 — cluster “casino / reno / peppermill”
1. Wife's fav place to go slot maniac. Did you know that the IGT machine factory in town often tests the newest machines here. Most of the time it means you win while they figure out how to make it pay out less. Love the Italian restaurant Romanza,. If we win we go there to get the …
2. Peppermill has done it again with service and accommodations that are second to none. Gaming at the mill (i only play 1 dollar or 25 cent slots) seems to provide better outcomes and I know reno. Spent a lot here and found the peppermill is the best. Food is some of the best and r…
3. Stayed at the Peppermill on our last leg of a cross country trip by vehicle after 3 weeks, this was a luxurious treat. I haven't stay here for a long time and the Tuscan style face-lift made it more enjoyable. Very comfortable room even on the 7th level. It's even better when you…
4. Best Poker room in Reno. We also stayed at the Hotel and would recommed it.
5. We spent six wonderful days at Hotel Mazarin, and I have to say that this has been the perfect hotel experience. Great room, wonderful breakfast, and every person on staff has been a joy to deal with. Very close to - but luckily not ON - Bourbon Street, and close to everything in…

### Block 4 — cluster “hotel / orleans / great”
1. We just returned from a girls' weekend trip to this hotel and I have nothing but praise. Our room was lovely, so clean and spacious (walk in closet!) We had a double queen suite. The hotel is about 1 mile from the FQ, but the walk is pleasant and the River Walk Outlet is on the w…
2. As promised, I came back and am glad I did. Try the parking across the street (12th), only $8 per 10 hour span on the weekends.
3. Excellent location!! This hotel sits kitty-corner from Independence Hall entrance, right across the street from Liberty Bell and a stones-throw from many other historical significant sites in Philadelphia. It is also an easy stroll to Penn's Landing, South Street and two blocks d…
4. This is an ideal family getaway stay in St. Louis. It is near the St. Louis Zoo, Grants Farm and the St. Louis Science Center. This is the perfect starting location for navigating through the busy streets of streets of the city. The rooms are very up-to-date in European style. On…
5. A pretty hotel located in New Orleans Art District. The location is very convenient to the convention center but is also walkable to other attractions! The hotel is so clean, and staff was very polite and helpful. The complimentary workout room is decent- two treadmills, 2 bikes …

### Block 5 — cluster “hotel / philly / city”
1. We had a fantastic stay at Hotel Monaco. We got a great deal on Hotwire - $100 a night! - a heavy discount on the normal price for a twin. The experience was so good however, that we'd be more than tempted to pay the full price if we're in town again, it was that good. Every staf…
2. This is one of the few hotels I have visited just to visit, and not to stay at. It's beautiful and really nice to just walk around and take it all in. My brother and sister have stayed here for an event, and they really enjoyed it, except for the high price tag. The convention ce…
3. Excellent location!! This hotel sits kitty-corner from Independence Hall entrance, right across the street from Liberty Bell and a stones-throw from many other historical significant sites in Philadelphia. It is also an easy stroll to Penn's Landing, South Street and two blocks d…
4. Easily my favorite hotel in Philadelphia. Well located in Center City and the rooms are exquisitely designed. Several times while staying here with my girlfriend, we've had the pleasure of dealing with a housekeeper named Sam. His service has been absolutely outstanding every tim…
5. We love Philly and go every New Years Eve. We have experienced many hotels in the area ... this by far was our worst. The location is ok, not great. Their are some restaurants ( El Rey, continental) within walking distance. (walking distance to us is 5-10 blocks). The parking is …

### Block 6 — cluster “room / hotel / stay”
1. Airport pickup never showed up after an hour and multiple calls. Hotel stopped answering the phone. Very poorly managed.
2. The worst We stayed at the hotel and went to the very empty bar that first night. My cousin got married the second night and the **very large** manager?? wouldn't let my wife upstairs --too much to drink? But she let up every heifer in the club,but my wife on a Friday night--No w…
3. My home away from home when I travel on business to Indianapolis. The rooms are large, stylish, and always very clean. I appreciate the different options for food on site as well. The staff is always very helpful and professional from the moment you walk in the door until you lea…
4. Booked a 1 King Bed Executive on the 24th floor and the view of the city from that high above was amazing. However, I had a couple of problems with the professionalism of the staff which is why I rated three stars instead of five. I can't help but make this personal and say that …
5. My favorite hotel!!!! along with their sister Hotel the Avalon next door. I live about a hour away but come as often as I can because there's so much to do here. The pool, food and drinks are amazing, along with their extremely friendly staff. Perfect for a special occasion, birt…

---

<details><summary>Answer key</summary>

- Block 0: intruder is #5 (from cluster 4, “hotel / orleans / great”)
- Block 1: intruder is #3 (from cluster 2, “jw / hotel / indianapolis”)
- Block 2: intruder is #5 (from cluster 5, “hotel / philly / city”)
- Block 3: intruder is #5 (from cluster 4, “hotel / orleans / great”)
- Block 4: intruder is #3 (from cluster 5, “hotel / philly / city”)
- Block 5: intruder is #2 (from cluster 1, “nashville / hotel / opryland”)
- Block 6: intruder is #3 (from cluster 2, “jw / hotel / indianapolis”)

</details>

## Qualitative inspection — `two_stage__1000__s42`

_Samples are random cluster members (not centroid-nearest). Ask per cluster: can you name this in one phrase, and do the samples fit that phrase?_

**Noise:** 263 documents (26%) unassigned.

### Cluster 0 — “room / hotel / great” (294 docs · avg 3.84★, label: terms_fallback)
**Top terms:** room, hotel, great, nice, stay, clean, staff, place

- pros: -Location is great. 5 minute walk to Knitting Factory music venue, record store, plenty of restaurants and shops -Sweet pool and hot tub -large room with huge windows and a great view of downtown -$10 pet fee -1PM check out time -nice fridge cons: -toilet appeared to have n…
- Don't be deceived by the fact that it is a little older, this place is meticulously maintained!! Every corner is clean and beautiful fixtures in the rooms. The room we had was small, but very nice. We were only in town 1 night, so bigger wasn't necessary. The courtyard is also be…
- Very nice hotel for this price range. Clean and comfortable. Friendly staff.
- Comfortable room with a awesome guitar swimming pool. Short trip to downtown. Rooms are clean and staff are friendly.
- I enjoyed my stay here, despite the horrifically slow elevators. Come to think of it, I think only one (of three) was working! Now I know why people have weddings here-- I love the decor/look. The staff was extremely helpful and friendly. Free internet and printing in the lobby (…

### Cluster 1 — “hotel / room / stay” (79 docs · avg 4.05★, label: terms_fallback)
**Top terms:** hotel, room, stay, city, nashville, staff, like, great

- We booked this hotel through Priceline.com's name your own price. This which was the first time we had used this service, so I was a little leary about how smooth it would be. I was impressed with the ease of check-in, the indoor pool, and the rooms were nicely decorated and very…
- The only knock on this place is that they didn't have all the pools open during Memorial Day weekend. Other than that the hotel provides a very friendly environment that's good for adults and families as well plenty of places to eat inside the resort and the atrium was amazing an…
- Great external look, amazing five-star staff, and outstanding meeting space and capabilitiesmake this place very worthy of five-stars, but issues with the functionality of meeting space accommodations render this hotel four stars. Riding to the hotel in an Uber, i didn't know wha…
- My home away from home when I travel on business to Indianapolis. The rooms are large, stylish, and always very clean. I appreciate the different options for food on site as well. The staff is always very helpful and professional from the moment you walk in the door until you lea…
- Lovely boutique hotel! Room was pleasant - a little small but certainly enough space for 2 good friends. Very convenient to almost anywhere in the city. Good relationships with neighboring businesses such as restaurants, gyms and parking facilities. Had a reunion weekend with a c…

### Cluster 2 — “room / hotel / desk” (62 docs · avg 1.68★, label: terms_fallback)
**Top terms:** room, hotel, desk, told, did, called, manager, staff

- The kid at the front desk refused to get a manager or supervisor after I asked. The kid named derick told me I was touchy...very rude. Turns out the supervisor was nearby and I talked to him. (They all gave me different info so I gave up). The AC did not work properly in room. Th…
- This hotel was nasty. There are termites and ants crawling out of the floor and up the wall in droves outside our room door. Don't stay here. I uploaded a couple of pics of the bugs outside our door. So gross.
- This is your typical large conference hotel. I came here for business and stayed six nights. I found that the staff was not customer service centric - probably because their occupancy was so high and maybe they felt like they didn't have to earn our business. Here are the things …
- I did not want to post this on Yelp. But after a surprisingly unconcerned and deflective conversation with the management of the Sandpiper Lodge here it is: I stayed in room 239 at the Sandpiper Lodge in Santa Barbara on 5-20 and 5-21- 2011. Today, 5-27-2011, my physician has inf…
- Let me start with the positive. The staff and customer service here is exceptional. They will go out of their way to make things right for you and accommodate your needs. However, the hotel facilities they have to work with is in dire need of renovation. The shower in our room dr…

### Cluster 3 — “oysters / good / food” (78 docs · avg 3.81★, label: terms_fallback)
**Top terms:** oysters, good, food, shrimp, place, service, ordered, great

- I really wanted to give this a higher rating. I've heard great things about their cocktails and small plates. My friend and I each ordered two different cocktails. I was underwhelmed by all of them; Time Enough For Love was the best of the four we ordered between the two of us, i…
- One of my favorite places ever, to grab a bite to eat. Small plates are the theme here.. But that does NOT mean you have to share ;) Their short rib panini's are to DIE for.. Not to mention that complimentary black truffle popcorn Atmosphere is laid back, modern, and comfortable.…
- Huge oysters when I was there. Great service! Had a muffuletta (half). It was all very tasty!
- Food was a 3.5 or so, but our server Kiki pushed our rating up to 4. Oysters on half shell were good (but so damn big!). BBQ oysters were really tasty. Turtle soup - solid and spicy. BBQ shrimp - wife loved it, I thought just ok.
- I don't eat raw oysters by my husband does and this was an experience. Kntrell and Frankie were the best oyster shuckers that I've seen in a very long time. I had the New Orleans style BBQ Shrimp and then went onto Fruits De Mer Platter, My husband ate the oysters and ate the res…

### Cluster 4 — “hotel / quarter / orleans” (86 docs · avg 4.17★, label: terms_fallback)
**Top terms:** hotel, quarter, orleans, french, great, bar, new, room

- Disclaimer: I'm writing this review for the common areas of the hotel, which I've experience during the past two "Tales of the Cocktail" events. The rooftop pool is small but offers a truly spectacular view of the city. Lounging poolside during TOTC made me feel like I was at an …
- I had a very enjoyable stay here while attending a conference. It's an older hotel but very well kept up. The room was clean, though a bit on the small side, and everything worked. The staff was great, very friendly and attentive. It was nice being in the heart of the French Quar…
- Hotel Monteleone is in the perfect NOLA location - immediately at the beginning of The French Quarter. It is conveniently located to all the cool & crazy "happenings." We had a "traditional" style room, which was small, but beautifully decorated with old world charm. I referred t…
- Nice. Great room; Great price. Had a little problem with the hot water the first night, but it got better. They offered a free night due the problem. Nice location of the French Quarter. The fourth floor concierge lounge was not much.
- To begin with, our experience at Hyatt was pretty awesome. All of the employees we engaged with were friendly, kind, and eager to help us out. The room we had was facing the Mississippi River and the Arch, which was definitely an added bonus! The bed was comfy and the bathroom wa…

### Cluster 5 — “casino / reno / peppermill” (98 docs · avg 4.1★, label: terms_fallback)
**Top terms:** casino, reno, peppermill, room, place, rooms, best, stay

- We stayed at the Peppermill Hotel for one night on our drive to Idaho. The hotel was clean and neat. The room was spacious. All in all a nice place to stay and the price was excellent. The front desk personnel were very nice but slow. We waited in line for a fairly long time comp…
- I'm in side the Hardrock standing at the Noodle Bar (food is wonderful). The problem is I am standing , again, with the 6 other people waiting for one of the 16 seats they have for a very, very popular restaurant. Although I'm sure the casino did not expect this place to be so po…
- Oh, Peppermill...you used to be my home away from home but lately, I just need to say a few things... I started staying with you back in 2008. A coworker recommended your lavish Tuscany suites and what a sweet deal it was. I enjoyed many fun times staying with you... from weekend…
- I don't gamble, so this review is for the pool area, which I visited recently with friends who were staying in the hotel. WOWZA! When can I come back? The pool area consists of three pools, all of which are perfectly warmed, and two jacuzzis. These are surrounded by the cushiest,…
- Always a wonderful experience. We never stay anywhere but the Peppermill because the hotel is consistently outstanding. The rooms are beautiful, the staff is extremely professional and the restaurants are excellent. The Peppermill gives you the Las Vegas experience without the lo…

### Cluster 6 — “beach / hotel / great” (40 docs · avg 4.35★, label: terms_fallback)
**Top terms:** beach, hotel, great, room, tampa, place, santa, stay

- Stayed here for a bachelorette party. This place got the job done. Unfortunately, all the hotels in Santa Barbara are pretty freaking pricey. A lot of the other hotels started at $400+ /night. This ain't Vegas where you can get something pretty cushy for $250. This seemed like a …
- The tiki bar has a great outdoor music venue, and the inside bar can be a fantastic time if you want it to be. They had on stage some of the most talented musicians I had ever heard. They were very versatile, playing all kinds of music from rock, to dance, to hip hop, to oldies. …
- We found this hotel on hotel tonight for a total of 123 a night. We had previously stayed at the Castillo Inn at the beach the night before and had a not so great experience, check out my review for that! Anywhoo, the price was cheap for the night of, I would recommend the app to…
- Cute little motel located in a easy walk to the beach location of Santa Barbara. Although the room was small, it was well appointed and newly upgraded featuring nice decor and bathroom amenities. They also offer a small continental coffee that included tasty coffee, juices and mu…
- The Aloft Downtown Tampa is a great hotel in a perfect location right on the Riverwalk in the downtown area. I would recommend this spot for travelers of all types. Tampa is really growing into a neat city, and this hotel is right near a lot of the cultural action in the area. Th…


## Intruder test — `two_stage__1000__s42`

_Each block shows 4 documents from one cluster and 1 from another, shuffled. If the intruder is not obvious, the boundary between the two clusters is not meaningful. Answer key at the bottom._

### Block 0 — cluster “room / hotel / great”
1. Nice lounge, cool atmosphere great lighting. Beds are low and you may bang you legs up, great location.
2. Very nice hotel for this price range. Clean and comfortable. Friendly staff.
3. pros: -Location is great. 5 minute walk to Knitting Factory music venue, record store, plenty of restaurants and shops -Sweet pool and hot tub -large room with huge windows and a great view of downtown -$10 pet fee -1PM check out time -nice fridge cons: -toilet appeared to have n…
4. Comfortable room with a awesome guitar swimming pool. Short trip to downtown. Rooms are clean and staff are friendly.
5. Had a great stay at this hotel right in the french quarter. convenient walking to everything on bourbon street and beyond. their continental breakfast had a large variety of items and was free per your stay. great comfy beds. A huge plus was the non-smoking hotel that doesn't all…

### Block 1 — cluster “hotel / room / stay”
1. Just stayed here for a conference for three days and nights. What a lovely hotel. Very clean and sleek and the staff was friendly, knowledgable and professional. Everyone had a smile and was willing to help. The food was amazing and the Market Place buffet had great choices, all …
2. If I were to visit the Philly area (even though it's about 30 minutes away) again I'd stay here. It was clean, bright, and very well maintained. My bill was around $133 for a suite on a Friday night. I left early in the morning so I skipped the complimentary breakfast.
3. Let me start with the positive. The staff and customer service here is exceptional. They will go out of their way to make things right for you and accommodate your needs. However, the hotel facilities they have to work with is in dire need of renovation. The shower in our room dr…
4. My home away from home when I travel on business to Indianapolis. The rooms are large, stylish, and always very clean. I appreciate the different options for food on site as well. The staff is always very helpful and professional from the moment you walk in the door until you lea…
5. Great external look, amazing five-star staff, and outstanding meeting space and capabilitiesmake this place very worthy of five-stars, but issues with the functionality of meeting space accommodations render this hotel four stars. Riding to the hotel in an Uber, i didn't know wha…

### Block 2 — cluster “room / hotel / desk”
1. Okay lets start from the beginning. 1) When i first go to check in the young lady at the front desk is on the phone tying to figure out where her lunch is and why its taking so long to get to her (not sure why i needed to know this) I go to check-in and since i paid for the room …
2. I stayed at the Hotel Monaco on a recent visit to Philadelphia and I was impressed with almost everything. Everywhere I went in the hotel itself, staff were happy, friendly, and welcoming. This wasn't the case only with me and it did not come across as fake or forced. I am a peop…
3. This is your typical large conference hotel. I came here for business and stayed six nights. I found that the staff was not customer service centric - probably because their occupancy was so high and maybe they felt like they didn't have to earn our business. Here are the things …
4. BEWARE ....this hotel has a case of bedbugs. I woke up at 1am scratiching my arm and I saw welts. I get out the bed turn on the light and I see one bug dead and one bug moving around in the bed. I told the night guy at the front desk to look at my video and he didn't want to see …
5. The Peppermill is the WORST casino in Reno -- in terms of atmosphere. The carpet and lighting make me dizzy and feel like I have to vomit just walking in. The ventilation system is not up to par, because the smell of smoke is so strong, everywhere in the casino. I won't go to any…

### Block 3 — cluster “oysters / good / food”
1. Came here for a late night dinner with friends. We shared a half dozen raw oysters and a half dozen chargrilled oysters. Ohhh myyy goodness!! Simply the best bivalves I've ever had...ever! While the chargrilled oysters were amazing, I preferred the raw oysters with a squeeze of l…
2. The food here is amazing and our server Geri is one of the best servers I have ever had. The char grilled oysters are a MUST. The reason I gave them 4 stars is because the manager was rude. Initially we asked if we could move our table so we can see the parade and he responded sa…
3. Ryan Watson is the greatest bar tender in the place. He knows his shit and can make a mean Bloody Mary. You are welcome!
4. Food very good, great service and a perfect view. I recommend the crab cake and bruschetta! But the coffee is terrible. $5 and it is just water that are colored black. Very disappointed because I love coffee and have tasted a lot better for less money.
5. Great history in this hotel. Rooms are spacious and well decorated. Bathrooms are a little cramped, but that's mostly due to the architecture of the building. Centrally located to anything you'd want to do in the Quarter, but far enough away from the unsavory elements on Bourbon …

### Block 4 — cluster “hotel / quarter / orleans”
1. Overall, this is a nicely appointed hotel with nice rooms, good service, and good amenities. The exterior and facade is beautiful as is the common space in the hotel. Carousel bar is nice and unique--definitely best hotel bar in NOLA (loews swizzle stick i like too), but is often…
2. Favorite Nola hotel! Great history and antiques, comfy beds, very clean. The pool area is great though it can get a bit crowded. We were there during jazz fest, and several band members and their relatives hung out at the pool.... fun just eavesdropping from the next chaise loung…
3. By far one of the most Luxurious Hotels in Reno! My favorite is the pool and spa. A must see for anyone visiting the Reno area!
4. I stayed here through a special on expedia.com during Halloween and I must admit I wasn't expecting much..that is until I arrived. The hotel is gorgeous, has a roof top pool and is right near the action. Honestly our room was facing a brick wall but the bed and shower made up for…
5. When I'm in the quarter, I always like to stop in at Flanagan's. It's one of those bars that's off the "beaten path" and a nice escape from the hustle and bustle of Bourbon (if you're a tourist) or even Decatur (if you aren't). The bar is open 24 hours and the bartenders here are…

### Block 5 — cluster “casino / reno / peppermill”
1. I'm in side the Hardrock standing at the Noodle Bar (food is wonderful). The problem is I am standing , again, with the 6 other people waiting for one of the 16 seats they have for a very, very popular restaurant. Although I'm sure the casino did not expect this place to be so po…
2. This hotel is awesome!! Very close to Philly and tons of restaurants and shopping. The room was clean and comfortable. Got a great night sleep. The pillows and the bed was great. The hotel was quiet and the staff were extremely friendly and helpful. I would definitely stay here a…
3. My boyfriend and his friends go every year for his birthday, and it's always a great place to stay with a large group even with kids. They've got everything from arcade, gym (with a room to do yoga/stretching etc!), pools, spa, club etc. I'm not much of a gambler but definitely e…
4. This is my favorite steakhouse in Sparks/Reno area. A great hidden treasure. Full of suprise and what a great dining experience. So far a cut Above the rest of the areas steakhouses.
5. Been coming here with my brother for 10 years and will continue to come here for many more I hope. Get a players card, it's free, and they don't care how much money you play with just how long you play. If you have a card you will get amazing comps like free rooms and dining disc…

### Block 6 — cluster “beach / hotel / great”
1. I love this place, it has to be one of the best values in Santa Barbara as a place to stay. 1. You are literally 2 blocks away from the beach, walking to local wine bars in the area is 10 minutes, walking to downtown State St. is roughly 10-15 minutes, and the best local bar in t…
2. Recently my husband and I stayed at this place and it was one of the better hotels that we've stayed at in the US. I had everything that a guest would want from a large clean room with a flat screen TV with lots of channels to a pool and fitness center. The room we got was huge a…
3. WORST HOTEL EXPERIENCE EVER! We arrived at the hotel around 3:00am after driving 12 hours from TX. The first thing we saw were a lot of police cars parked out front, apparently their were mutiple loud parties going on in the hotel that night and the police had to be called. We we…
4. I recently spent 2 nights here and have stayed a couple of other times too. I like the place, the grounds are nice, the pool and spa are great after a day at the wineries. All the rooms have a balcony or patio to relax on, they have a fire pit to enjoy a glass of wine by. The roo…
5. We got free tickets to the W&W show at the tiki bar this past Sunday. Our expectation were pretty low being we've never been to a party in Clearwater but we were pleasantly surprised. While there were tons of people there was still plenty of room to move around and places to sit …

---

<details><summary>Answer key</summary>

- Block 0: intruder is #5 (from cluster 4, “hotel / quarter / orleans”)
- Block 1: intruder is #3 (from cluster 2, “room / hotel / desk”)
- Block 2: intruder is #5 (from cluster 5, “casino / reno / peppermill”)
- Block 3: intruder is #5 (from cluster 4, “hotel / quarter / orleans”)
- Block 4: intruder is #3 (from cluster 5, “casino / reno / peppermill”)
- Block 5: intruder is #2 (from cluster 1, “hotel / room / stay”)
- Block 6: intruder is #3 (from cluster 2, “room / hotel / desk”)

</details>

## Qualitative inspection — `bertopic__1000__s42`

_Samples are random cluster members (not centroid-nearest). Ask per cluster: can you name this in one phrase, and do the samples fit that phrase?_

**Noise:** 356 documents (36%) unassigned.

### Cluster 0 — “casino / reno / peppermill” (90 docs · avg 4.24★, label: terms_fallback)
**Top terms:** casino, reno, peppermill, best, vegas, play, area, place

- Tuscan suites are sweet! Amazing rooms for very little cost. They went all out, finally something nice in Reno. Bring on the Vegas glitz!
- The Peppermill is always making upgrades to their hotel to make it look better. But if you're a non smoker, good luck on making it through without getting second hand smoke cancer. They used to have these portable air purifiers all over the casino but they have steadily disappear…
- Hotel rate from e-mail were very generous for the great room we had. Casino and restaurants were fun and food was excellent. Bright atmosphere in the casino.
- Peppermill has done it again with service and accommodations that are second to none. Gaming at the mill (i only play 1 dollar or 25 cent slots) seems to provide better outcomes and I know reno. Spent a lot here and found the peppermill is the best. Food is some of the best and r…
- I go to this casino very often and for long time now.. I'm very upset I need to pay $10 every time I go there for parking.. it used to be free and it should be free, for a place like that, that profits really nicely... it should definitely be complementary. I understand that now …

### Cluster 1 — “bar / orleans / new” (77 docs · avg 4.23★, label: terms_fallback)
**Top terms:** bar, orleans, new, hotel, quarter, french, great, nola

- 2nd trip to Nola and this was on my list. Old world charm meets modern day living. My 3rd trip to Nola and they had Mardi Gras decorations all over the lobby. Beautiful 12 ft ceilings designed in a Beaux-Arts-style boutique Hotel. When you walk through the doors the focal point i…
- Overall, this is a nicely appointed hotel with nice rooms, good service, and good amenities. The exterior and facade is beautiful as is the common space in the hotel. Carousel bar is nice and unique--definitely best hotel bar in NOLA (loews swizzle stick i like too), but is often…
- This place was the perfect location, the perfect price, and the perfect choice for our stay in New Orleans. If you're looking to stay in the French Quarter and need a place to sleep this is the perfect hotel. The staff is extremely friendly, they had a free breakfast, and the hot…
- Was in New Orleans for business. Stayed here for 5 nights at 199 a night. This place was definitely not worth it. Tiny room. Maybe 90 square feet. A view of a concrete wall about 3 feet from my window. Old carpets in halls and in rooms. Definitely not worth 199/night. Unfortunate…
- I stayed here through a special on expedia.com during Halloween and I must admit I wasn't expecting much..that is until I arrived. The hotel is gorgeous, has a roof top pool and is right near the action. Honestly our room was facing a brick wall but the bed and shower made up for…

### Cluster 2 — “oysters / good / shrimp” (74 docs · avg 3.77★, label: terms_fallback)
**Top terms:** oysters, good, shrimp, food, ordered, service, place, bourbon

- This place was pretty good. We came here on a Friday evening right before 6. Happy hour was just about to end, but we were able to get $1 oysters on HH. I ordered the jambalaya pasta and my bf ordered a pound of crawfish. It took a really long time to get our food. My food came f…
- Went to NOLA over the weekend and stayed at the Royal Sonesta. Desire Oyster Bar is one of many restaurants they have on site. We tried the place because it was super convenient and wanted something other than the continental breakfast included with our stay. Even if not staying …
- As someone from out of town, all I could look forward to was the food. And let me tell you it was a disappointment. I read many good reviews about this establishment. If I could give them no stars I would. Could not run out of there fast enough. Their half peeled shrimp was liter…
- The raw oysters were good, although I noticed the shells were dirty. I'm all for freshness but gotdamn can you wash them? That weirded me out just a little bit. The bbq shrimp was great, and very filling. I don't remember what my fiance had, but he ate all of it happily. The hurr…
- Ate at Desire two times, once 2 years ago and 2 weeks ago. Both experiences were picture perfect. The service is impecable and the food followed in suit. Dont let the fact that its dead smack on Bourbon Street put you off.......Desire is a Jewel....Guaranteed. Note: Remember that…

### Cluster 3 — “hotel / great / nice” (59 docs · avg 4.25★, label: terms_fallback)
**Top terms:** hotel, great, nice, location, friendly, rooms, stay, breakfast

- This is best described as country chic at it's best, it's old and probably needs a serious update, it has an old smell to it but clean and the staff is great BUT.... it has 2 restaurants on site with live music and a pool with cabanas indoors, to my surprise. It's an atrium style…
- Due to the look (and price) of this hotel, top notch is important to their brand. The hotel was able to accommodate over 3000 employees from all over the country seamlessly. They served us very well and even decorated it to the 9's!
- This is totallly one of the best hotels in the city. The rooms are heavily appointed, and the halls are too. You can't go wrong here.
- I am so excited about this hotel. It is really a hidden gem. We arrived at 8:30 am, planning to store our luggage until check-in time. The very friendly lady at the front desk, gave us a room immediately. She was very kind. They had a guy take up all of our luggage. He was very f…
- Good hotel with a great location... Not the "Roosevelt " but good for a few days. Be careful of the $40 a day parking fee... it added up quickly, it seemed high to me for a "Hampton Inn" property.

### Cluster 4 — “room / smell / sheets” (52 docs · avg 1.73★, label: terms_fallback)
**Top terms:** room, smell, sheets, hotel, like, clean, towels, smoking

- My Room was disgusting. The odor when I walked in was terrible. They had put a very large air filter in the living room to try and mask the smell. The carpet looks as if it was molding in the bedroom against the wall - looks to be the shower wall. Must have leaked. No iron had to…
- This hotel was nasty. There are termites and ants crawling out of the floor and up the wall in droves outside our room door. Don't stay here. I uploaded a couple of pics of the bugs outside our door. So gross.
- This was the WORST HOTEL STAY EVER!! asked for a Non- smoking room. Went to that room and it smelled so bad! Asked for another room and that room smelled as well. Not that clean, bathroom was old. Not a place you wanna stay at. And they charge a fortune for crap!! I don't even wa…
- This is one of the best experiences I have had in an Extended Stay America motel. The manager comped my room for a minor mishap and I stayed another night. There was a broken drawer in the nightstand and the shower curtain was hung too high, like a LOT of hotels I have been in, a…
- Where do I start? We've been touring the midwest staying exclusively at Homewood Suites over 8 nights in cities ranging from Dallas to Memphis, Chicago, Lexington and now Nashville. This place is a dump. The rooms are old and worn. The kitchen is falling apart, trim falling of th…

### Cluster 5 — “told / room / hotel” (47 docs · avg 1.83★, label: terms_fallback)
**Top terms:** told, room, hotel, desk, charged, did, asked, manager

- Beautiful. Booked a casita last week Fri-Mon. Accommodations were roomy and clean. Breakfast provided every morning was yummy. However things took a turn Sunday afternoon after my wife and I came back from our outing. Our keys had been deactivated. I went to the front desk and th…
- we haven't even stayed here yet, and this establishment has already created a bad impression! our family booked a 3 night weekend over the 4th of july via hotwire and i called the reservations desk to confirm. the hyatt person was supremely snotty (thanks, susan) and kept interru…
- I was so excited for this trip. I hadn't taken a vacation in years. I booked the king size bed with whirlpool tub for some romantic relaxation with my boyfriend. We were visiting Tahoe but decided to stay here specifically for that room. We check in, all is well and we get to our…
- Cancellation at this hotel sucks. I booked the hotel through booking.com and had to cancel because my mom had cancer and I was the only one who could take her to appointments. I made the reservation in October of last year and cancelled in November. My stay wasn't until March. I …
- This review is for the bar inside of The Saint Hotel. I went there yesterday with a girlfriend to order a drink while waiting on a friend. The bartender (Megan) was rude, unprofessional, and unpleasant. My friend ordered a French 45 and I ordered one of the "Favorites" the Pimm's…

### Cluster 6 — “room / hotel / bed” (27 docs · avg 3.81★, label: terms_fallback)
**Top terms:** room, hotel, bed, nice, clean, comfortable, pillows, king

- A very nice hotel and a good value. ($175/night) The king bed was primo, the room tastefully furnished. The rotating Carousel bar is fun and the pool at the top of the building is a great way to cool off on a 95 degree day.We stay there given the great location in the French Quar…
- I stayed here 3 nights for a work conference. Work covered a regular Peppermill tower long and I upgraded to a Tuscany king, but they were sold out of those and gave me a double king in Tuscany tower. That was fine with me, although the hotel appeared far from packed with people.…
- The hotel is beautiful and clean and the Carousel Bar is fun. My only complaint is that when we realized just how small a fill size bed is, there was no option for an additional cot. Seating was minimal in the room with one chair, but we didn't spend a lot of time in the room any…
- You see that picture of the hotel? I stayed in THAT ROOM. I stayed there a few years ago on a vacation, and they upgraded us to the top floor -- it was the day before Thanksgiving, and it was EMPTY. For the end of the vacation, we (me and the ex-) were tired, and this was the per…
- Nice hotel, room was very nice and clean, staff nice. Reason for the one star is the elevator. DO NOT get a room by the elevator, you can hear it and it is very loud with lots of banging. Unless you like a loud thud every 20-30 seconds, do not stay here. They put us in a room by …

### Cluster 7 — “city / philly / hotel” (26 docs · avg 4.46★, label: terms_fallback)
**Top terms:** city, philly, hotel, philadelphia, center, room, great, independence

- The place is straight out of an Escher etching. You know, with all those stairs that go nowhere? I discovered this maze of stairways when attempting to find a cup of coffee and maybe a muffin to go somewhere on the premise. I never found one. The convention facilities are luxurio…
- The Loews Philadelphia is what I imagined Gotham City looks like. The hotel is Art Deco and absolutely beautiful. We stayed over New Years and the decorations were beautiful. It's centrally located so there are tons of restaurants and museums within walking distance. The views ar…
- Kimpton Hotels are among my favorite hotels. They are trendy and modern. The decor is beautiful. I love that they are pet friendly and don't charge extra fees. They have peanut butter flavored treats for your pet at check-in . They can also provide you with a loaner pet bed, wate…
- Summary: great location, great service, incredibly comfortable clean rooms. First rate. We stayed here for two nights when we were in town to work the election. Got a good deal through TripAdvisor using Booking.com . When we arrived, we were told that the hotel upgraded our room.…
- Lovely boutique hotel! Room was pleasant - a little small but certainly enough space for 2 good friends. Very convenient to almost anywhere in the city. Good relationships with neighboring businesses such as restaurants, gyms and parking facilities. Had a reunion weekend with a c…

### Cluster 8 — “great / restaurant / inn” (23 docs · avg 4.61★, label: terms_fallback)
**Top terms:** great, restaurant, inn, clean, definitely, nice, place, gardens

- This is a great place to stay. I picked it because of the full kitchen, it was very helpful. We didn't spend as much money on food. Its very clean and cute!
- My Aunt and I stayed here for a few nights in December. The location is GREAT being very close to State St., pier, zoo etc.. The staff was AMAZING!! The girls who work in the front were extremely helpful, even offering us local family owned options for dining. The chef, Jenny doe…
- I am in love with this place! ! A charmingly friendly and helpful staff. Incredibly diverse excellent spread for breakfast. Chocolate covered strawberries as a treat in the room. Great location across from Redding Terminal. Great happy hour in the hotel bar!!
- Overall a fun place to be! Played some slots and had dinner at the cafe. There was live music but because of that you couldn't really talk at dinner. Food was mediocre though.
- I came back to Santa Barbara for a little pre-Mother's Day weekend getaway. My mom had never been here so it was perfect. A few months before the trip I knew I just had to reserve a room at Castillo Inn because last time we were very happy with the room, amenities and location. T…

### Cluster 9 — “great / rooms / stay” (22 docs · avg 4.59★, label: terms_fallback)
**Top terms:** great, rooms, stay, comfortable, room, nice, clean, beds

- Artsy, modern rooms and decor. Bathroom was super clean and modern. Rainfall shower all large plate tiles great place with awesome views of the city or river depending on where your room is. Trendy bar downstairs. With a Dj spinning hip music lots of places to sit and hang out. G…
- Very nice, spacious little apartments, clean, attractive, plenty of appliances, enough electrical outlets, great wi-fi, and quiet inside the apartment, I would definitely stay again!
- Don't listen to any of the bad reviews from previous ownership! The Rathbone Mansions completely surpassed our expectations! The room was clean, quiet, and comfortable--updated, but still historic and charming! The TV appeared to be brand new, the AC worked great, the shower wate…
- We have stayed here four other times. The rooms are well appointed and very well. Coffee is available in the lobby. Water with. Citrus is always in the lobby. Valet parking in a secured site on premise. The hotel is on the national register of historic places. The courtyards are …
- This place was great! It's a nice old building and the bricks give the rooms a nice feel. It's also close to the French Quarter. The A/C worked great and the room was comfortable. The problems I've had were fairly minor. The floors squeak really bad. You can't breathe without the…

### Cluster 10 — “nashville / hotel / downtown” (20 docs · avg 3.6★, label: terms_fallback)
**Top terms:** nashville, hotel, downtown, shuttle, time, opryland, resort, place

- I have tried nearly every Hilton chain hotel in the Nashville downtown area and am running out of options. I travel extensively, spending at least 200 nights per year in hotels, with most of those being the Hilton chain. I've come to expect a certain level of service. The Garden …
- I have been coming here for years and it only gets better every time I come here. It's close to Nashville and all the fun that city has to offer. A great place for a vacation.
- When in Nashville ...... Stay here Back a year later and Andrea took great care of us Never would stay anywhere else!
- My mother booked this hotel for a 1 night stay so we could take my 5 nieces and nephews to Nashville shores and this location was literally 5 minutes away. We had some issues with the owners who ran the place. They weren't the best or friendliest. And the morning of check out, th…
- Very nice. Pricey? Sure, but well worth it. I don't know of a whole lot to do in Nashville, but while I'm there on business, I love staying at the Opryland. It's really one of those things you just have to experience.

### Cluster 11 — “noise / sleep / bed” (18 docs · avg 3.11★, label: terms_fallback)
**Top terms:** noise, sleep, bed, night, room, hear, didn, door

- The room we had was huge. It was a non smoking two queens. Very roomy in the general area. Bathroom very small and old. Smoke from the hall way made it in. Very bad. Price was very good. It served what we needed. Place is part of Peppermill so decor is kind of dark. If u need a p…
- Great property. Minimalist decorating. Bar and 2 restaurants onsite. The rooms get a 5 for sleep stars. Zero noise between neighboring rooms. It is a huge draw for a few local companies and whatever local convention is taking place (center adjacent to the hotel). The pillows rece…
- Be prepared to be unimpressed. Front desk lost our reservation. That should have been my first clue. Second is the musty mold smell that greets you at the elevator bank. Third is the ill maintained room. My stay here is due to a medical appointment for my son in the morning. My f…
- Three night stay. Not plush but not a total dump. East side was noisy from a factory air vent and cable was short on channels but right near airport and Pima air museum with several breakfast spots (Waffle House and Denny's) close.
- The room was rundown and right next to a watercooler. There is literally no soundproofing so I had the pleasure of listening to teenage girls pound on different doors all night long. I had to call security to deal with the noise issue and it even took them quite a bit of time to …

### Cluster 12 — “pool / water / got” (15 docs · avg 3.47★, label: terms_fallback)
**Top terms:** pool, water, got, room, time, use, bar, soft

- My fiancee and I stayed here over the Veterans Day holiday because we had a coupon. Apparently everyone had the same genius idea I did because when we got there the line was pretty long. We waited about 30 minutes to check in but it was worth the wait. We stayed in room 1534 in t…
- I've stayed in the old tower in a jacuzzi suite and in the new tower in a regular suite. They are excellent rooms with plenty of attention to detail. The rooms are comfortable and if you get the choice, stay on the Mt. Rose side (it's a great view to wake up to). Be careful with …
- Stayed here during April which was too early to use the outdoor pool. Still the rooms were nice and the staff were friendly. Also, it is not difficult to use the pool even if you are not a guest here...
- We stayed here for 3 nights late may. Great hotel. The rooms are clean and the staff are helpful. We stayed in the corner room which had a great view. The pool is on the roof. The water in the pool was very murky and very unappealing to swim In. And pool bar was awful. The roofto…
- Booking it for New Years Eve, this hotel was chosen mainly for its location, which is amazing. You are only a block from the river, particularly Penns Landing and Spruce Harbor Park. It has great views of the Ben Franklin Bridge and the River. Unfortunately, this was the only pos…

### Cluster 13 — “staff / nice / friendly” (14 docs · avg 3.21★, label: terms_fallback)
**Top terms:** staff, nice, friendly, rooms, amazing, spaces, unit, clean

- From the moment we arrived the front desk staff was feiendly and welcomjng. They provided us with information on what to do close by the hotel and provided us recommendations amd tips for our hike at Sabino Canyon. The service throughout our stay was amazing. The Gringo Grill sta…
- Great location and nice staff. Housekeeping is ok. They made our beds but didn't pick up the towels on the floor and didn't replace.
- Nice and clean! The staff was friendly and made great suggestions on places we should try. Very nice rooms. Only complaint was they said they had a pool but didn't mention it was at the Marriott 2 blocks away.
- Very nice deco friendly staff. Took a little staycation sometimes u need that. Room since and clean, had a wonderful view went shopping at the fashion mall. Nice
- Beautiful lobby, average rooms. Staff up front are very nice (as mentioned in previous reviews), but they don't have enough staff to keep rooms up. Not enough space in bathroom to hang up towels and cleaning staff don't get around until 3 or after.

### Cluster 14 — “breakfast / nice / room” (13 docs · avg 4.23★, label: terms_fallback)
**Top terms:** breakfast, nice, room, coffee, star, pool, hot, located

- Nice stay hotel nicely decorated room OK nice comfortable bed after being out and about, paid $25 to park stated in/out but just used trolley to get around. Located about 4 blocks from Harrah's Casino and Bourbon Street. Breakfast was nice eggs oatmeal biscuits sausage patty, bre…
- The staff here is amazing. I stayed here for two weeks with a big group for work, and management, front desk, servers, and drivers were friendly and accommodating. There is a clean and decently equipped gym with tvs; the laundry room was equipped with 4-5 washers and dryers and a…
- Beautiful lobby and helpful staff all the way through from check-in to check-out. There are plenty of cabs outside if you do not want to take mass transit. If you do need the train or bus, there is a subway stop one short block away and numerous SEPTA bus stops up and down Market…
- Stayed here three nights on business. The property is located very close to the airport but you really don't hear it. The rooms are standard TownePlace Suite rooms with a refrigerator, stove, etc.. The area in which the hotel is located is quiet with a combination of other hotels…
- Star 1: Clean Comfortable room, and through the auspices of Kayak, one of the cheapest choices in town. Star 2: Safe surrounding neighborhood. The cheapest hotel choices are frequently in the dodgier areas of town... not in this case! Star 3: Climate control pt. 1: outstanding a/…

### Cluster 15 — “room / free / wifi” (13 docs · avg 2.92★, label: terms_fallback)
**Top terms:** room, free, wifi, airport, stay, sliced, breakfast, hotel

- Summary: good service, lacking amenities, had a good time, one stay is enough. First off, please don't see any of my complaints as reflecting poorly on the staff, all of which did a great job and deserve whatever employment they have. Most of the issues I had were beyond the staf…
- Cockroach in my room, wifi is slower than aol in 1996, private beach costs $10 a chair. I'll never come here again.
- Once again, I wish that I could give half stars, because this is really a 4.5 review. Simply put, we were very happy with almost everything about this hotel. Its location cannot be beat -- just across the street from the French Quarter proper, on a major bus line, and about a 7-m…
- Converted building exceeds Embassy Suite/Hilton Standards Located close to the airport in an area with apartment/condominiums, the property is a converted property with a Hilton/Embassy Suite Stamp so it does not have usual open atrium. As a result, the rooms seems like they were…
- There are so many wonderful things to say about this hotel: 1. Free Wi-Fi 2. Free Shuttle Downtown 3. Free Snacks 4. Cute Greeting (Post Card, Guitar Picks and Goo Goo Cluster) 5. Uber Helpful Staff (We had to leave a night early due to weather and they worked with Hotels.com to …

### Cluster 16 — “marriott / room / service” (11 docs · avg 2.55★, label: terms_fallback)
**Top terms:** marriott, room, service, hotel, terrible, breakfast, degrees, better

- The TV shows are $4.99 and they have commercials! What a cheesy way to make money and a sign of a less than classy hotel, particularly when you pay more than $150 a night. And there is NO complimentary breakfast, just an overpriced buffet, something even the cheapest hotels in Ca…
- Hotel is massive and has a lot for the family to do while I was at a convention. The rooms were a bit on the small side and we could hear everything our neighbors said. This was a problem because a bunch of college kids were on our hall and made a ton of noise. All of the restaur…
- Nice hotel and gorgeous views with the plants, lights and fountains. It's a Marriott property but without the Marriott customer service. $25.00 for a long line breakfast buffet for mediocre food. Parking is insane...even valet is a long wait. The ability to acquire tickets to eve…
- Marriott could care less about helping you. Wow. Love this hotel. Hate the mgmt Also. It's music city USA and u can't hardly hear music in the hotel
- Pros: Marriott points On Canal and close to Bourbon Very fast internet that worked throughout the hotel Cool vibe A robe (yes I like it in a Hotel lol) Cons: No movies available at all Fridge really tiny Terrible terrible linens Super flimsy towels AC numbers are meaningless.. Do…

### Cluster 17 — “tampa / hyatt / grand” (11 docs · avg 4.09★, label: terms_fallback)
**Top terms:** tampa, hyatt, grand, staff, hotel, downtown, experience, airport

- This hotel is located near the Tampa International airport. I stayed here last week and have stayed here several times over the past 8 years. This Embassy Suites is not like most Embassy Suites as it does not have the center atrium. The hotel is a standard high rise building. Bad…
- Stayed here for one night prior to cruising out. Having access to the shuttle to pick my group up from the airport, take us to a restaurant for dinner and Walmart was absolutely great. Richard the shuttle driver kept us in stitches. The room was clean and neat. The breakfast was …
- I stayed at the Grand Hyatt in Tampa for an evening back in September of this year and I have to say it was a pleasant stay. My daughter and I flew in from California on a late flight so we didn't arrive at the hotel until past midnight. The lady at the reservation desk was super…
- The Aloft Downtown Tampa is a great hotel in a perfect location right on the Riverwalk in the downtown area. I would recommend this spot for travelers of all types. Tampa is really growing into a neat city, and this hotel is right near a lot of the cultural action in the area. Th…
- The hotel seemed fine to me for the money, even with the resort fees. It's only 20 mins to downtown. I do think that some of the valets need more training though. Some of the them couldn't answer basic questions like "where is the spa entrance" (which is basically right there nea…

### Cluster 18 — “beach / hotel / palm” (11 docs · avg 4.27★, label: terms_fallback)
**Top terms:** beach, hotel, palm, pavilion, pool, great, small, stay

- This is my go-to spot on the Gulf Coast. Clean hotel, small but efficient rooms and nice secure feeling. Great courtyard with landscaped pool area and beach access is perfect, right next to the historic Long Pier and nearby the Seabird Sanctuary. Most of the rooms face inward on …
- Wonderful place to stay & enjoy some beach time! We stayed in room 202 & it was nice, spacious, modernized, & comfortable! The place in General has plenty of parking, security, private pool, private elevator, & a wonderful BBQ area with tables & chairs & most importantly in front…
- When I was very little I grew up on Clearwater Beach, living a short distance from this hotel, then known (I think) as the Gulf View Apartments. I would have figured this long-time beach fixture would have been demolished long ago to build some other high-rise hotel or condo, but…
- Our family of four stayed two nights at the Palm Pavilion Inn a couple of weeks ago. This was our first stay at this hotel but after reading reviews, we had a good idea what to expect. First of all, this hotel is NOT a resort. It's an older, small, budget friendly, 3 story hotel …
- We struck gold and found this 5 star gem on Yelp. Fabulous ambiance, 1/2 block from the beach, and to our great pleasure- quiet. Our room was a like a charming beach bungalow, with a fireplace, great lighting, mini kitchen, high end bed /bedding, and pillows that didn't make me m…

### Cluster 19 — “wedding / anniversary / linda” (11 docs · avg 4.55★, label: terms_fallback)
**Top terms:** wedding, anniversary, linda, night, end, husband, day, wonderful

- We recently spent our 19th wedding anniversary at the rebrand Fess Parker/Red Lion/ Doubletree hotel. It was a staycation and we were only there for 1 night. We'd stayed here before when it was a doubletree so we knew what to expect. When they learned it was are in a versary they…
- What a great crew, had my wedding there and it was perfect. Leila, Frankie great job, couldn't have asked for more.
- Wish there was an option for ONE STAR. An epic disappointment on my wedding weekend. To be continued...
- Absolutely one of the best experiences for myself and my family. This review is waaaaay overdue. We arrived to Terrell House for a wedding. The place is spectacular. Like stepping into the 1800s, except with all the modern accomodations (AC, etc). Linda, the owner is a sweetheart…
- What a great anniversary weekend we just had! Thanks to Steve Caputo, Sonia Collins and LeaLea. We felt like tourists (maybe even royalty?) in our hometown. The first of many staycations to come. We had such a nice time at the pool and relaxing with our side by side pedicures at …

### Cluster 20 — “staff / breakfast / room” (10 docs · avg 4.4★, label: terms_fallback)
**Top terms:** staff, breakfast, room, provided, welcome, recommend, hotel, stay

- My husband and I stayed for one night. Upon arrival, the check in was easy and fast but the front desk associate could have been a little more professional in the way she looked and handled our reservation. She did not offer any direction as to how to get to our room. She was alr…
- We just had a fairly large contingent arrive for a wedding from France and Los Angeles. Everyone was welcomed by the hotel staff and had nothing but compliments and accolades for the staff and accommodations. The complimentary hot breakfast was a huge hit. Sheri Caja, Director of…
- Staff is amazing here. So helpful and supportive when we stayed in 2015. They knew we had a lot of stuff to load in our car, plus two little kids, so they let us take our time. We were way past the checkout hour by the time we left, lol! Hotel room was always clean, house keeping…
- We had a really great stay overall at The Eagle Inn. We loved the location (very easy to walk to the beach and get to the main shopping area. We found the staff to be friendly and helpful with all of our requests. The kitchen staff even remembered my two year old's likes and disl…
- I am happy to give this a nice rebound of an update. I stayed her in mid April over Spring Break. I will keep it short, sweet and right to the point. The room was almost perfect, they forgot the trio of shampoo, conditioner and lotion. Everything from the front end, to housekeepi…


## Intruder test — `bertopic__1000__s42`

_Each block shows 4 documents from one cluster and 1 from another, shuffled. If the intruder is not obvious, the boundary between the two clusters is not meaningful. Answer key at the bottom._

### Block 0 — cluster “casino / reno / peppermill”
1. The Peppermill is always making upgrades to their hotel to make it look better. But if you're a non smoker, good luck on making it through without getting second hand smoke cancer. They used to have these portable air purifiers all over the casino but they have steadily disappear…
2. Always a wonderful experience. We never stay anywhere but the Peppermill because the hotel is consistently outstanding. The rooms are beautiful, the staff is extremely professional and the restaurants are excellent. The Peppermill gives you the Las Vegas experience without the lo…
3. Underwhelming hotel, overwhelming experience, overpriced everything, humid/smelly atmosphere, and loud. I don't understand why anyone likes this? Yes, they have fake waterfalls and real plants inside, but it's so crowded, it's HUGE (literally could take 20 mins to get to your roo…
4. Peppermill has done it again with service and accommodations that are second to none. Gaming at the mill (i only play 1 dollar or 25 cent slots) seems to provide better outcomes and I know reno. Spent a lot here and found the peppermill is the best. Food is some of the best and r…
5. Very nice deco friendly staff. Took a little staycation sometimes u need that. Room since and clean, had a wonderful view went shopping at the fashion mall. Nice

### Block 1 — cluster “bar / orleans / new”
1. We just spent 3 nights here in a Royal Salon suite. It was a nicely appointed, spacious, clean room. Not really a suite in that it was a single room with upgraded bath (separate jacuzzi tub and shower). The bathroom, extra space, and view were worth the 30 extra bucks per night (…
2. We just returned from a girls' weekend trip to this hotel and I have nothing but praise. Our room was lovely, so clean and spacious (walk in closet!) We had a double queen suite. The hotel is about 1 mile from the FQ, but the walk is pleasant and the River Walk Outlet is on the w…
3. Easily my favorite hotel in Philadelphia. Well located in Center City and the rooms are exquisitely designed. Several times while staying here with my girlfriend, we've had the pleasure of dealing with a housekeeper named Sam. His service has been absolutely outstanding every tim…
4. Was in New Orleans for business. Stayed here for 5 nights at 199 a night. This place was definitely not worth it. Tiny room. Maybe 90 square feet. A view of a concrete wall about 3 feet from my window. Old carpets in halls and in rooms. Definitely not worth 199/night. Unfortunate…
5. Went there to see the Nashville Yacht Club Band with some friends, had a great time. The atmosphere was perfect, I love the open bar and pool area connected. The bartenders name was Eckhard and he really made the experience worthwhile. I could tell they were understaffed and no o…

### Block 2 — cluster “oysters / good / shrimp”
1. Pleasantly surprised by this joint. We were expecting tired and average food but we had excellent service, flavorful food with excellent flavors. The roasted russet potatoes were to die for - perfectly prepared with a soft inside and nice crispy skin. Most rotating restaurants sp…
2. The food here is amazing and our server Geri is one of the best servers I have ever had. The char grilled oysters are a MUST. The reason I gave them 4 stars is because the manager was rude. Initially we asked if we could move our table so we can see the parade and he responded sa…
3. As someone from out of town, all I could look forward to was the food. And let me tell you it was a disappointment. I read many good reviews about this establishment. If I could give them no stars I would. Could not run out of there fast enough. Their half peeled shrimp was liter…
4. I had the worst duck that I have ever had. It was so tough that I could hardly cut into it. The cherry sauce was uninspiring at best. My wife had a curried sole, in which the turmeric was overwhelming. The only creditable thing was that they agreed to take the duck off my bill.
5. I've stayed at a lot of Tampa hotels and this one is one of my favorites so far. The rooms have microwaves, mini fridges, two room areas and great views of the downtown area. The on-site restaurant is decent and offers room service with good service. My gripes: wifi is only free …

### Block 3 — cluster “hotel / great / nice”
1. A very nice place with very nice people. We came with our twin baby boys for a holiday family gathering. There are a great many of us (I'm guessing roughly 20 staying at the hotel) and everyone enjoys the place. There are some quirky details worth noting... First that it had a na…
2. The hotel is beautifully done. It has a boutique feel and not like just a normal hotel like the Hilton. I had my birthday party there at the restaurant and it was so very nice. The service was wonderful and the rooms are beautiful. The air conditioner was pretty loud in our room …
3. Very modern, colorful decor which I enjoyed. Definitely not your average hotel. Many seating areas in the lounge, along with a bar and a pool table. They've a decent infinity pool that also features many seating areas for groups and modern pop music playing outside and in the lou…
4. Great location. Good value for the money. Clean rooms most of the staff is awesome. The regular chef in the morning is smoking great. Breakfast can't be better. The dining room in the evening not so good. Bartender has the personality of a slug. Waiter tries hard but is overwhelm…
5. This is the closest hotel to the airport and they offer a free shuttle which is nice. The nightly rate of $111 is definitely not worth it but my flight was canceled so I had no choice. Was definitely a better choice then sleeping at the airport. Management really needs to conside…

### Block 4 — cluster “room / smell / sheets”
1. Let me start with the positive. The staff and customer service here is exceptional. They will go out of their way to make things right for you and accommodate your needs. However, the hotel facilities they have to work with is in dire need of renovation. The shower in our room dr…
2. I had the pleasure of staying in this hotel for a night and I was impressed. The best thing about this property is the view from the room. By far the best thing the hotel has to offer. Parking was extremely expensive at this hotel, but it is expected. I had one issue with the hot…
3. GREAT place to stay! My husband and I spent our anniversary night here and the front desk (Jake) sent up a complimentary bottle of champagne and anniversary card. We stopped at the bar on our way out to dinner and the bartender gave us 2 drinks on the house as well. Great service…
4. Rooms were nice and clean. But staff was unprofessional and uncaring. At check in, I was immediately informed that someone from a neighboring hotel had been shot and killed and that the front desk clerk was nervous to be in the area. Arrived back to room after long day with 2 you…
5. Spider, spider everywhere! Best for the Halloween day ;-) We had to literally kill a few spiders inside the washroom and one nearby the bed. Worst hotel I ever lived... Even non-smoker room smell smokey, tiles and fixtures were broken (check the posted pics), and the water was dr…

### Block 5 — cluster “told / room / hotel”
1. We spent memorial day weekend here for a wedding. The facilities were fabulous. Our room was clean (a little on the smaller side, especially for the price we paid), but we stayed in Magnolia, which is such a HIKE to the rest of the hotel. I wouldn't mind walking everywhere if I a…
2. Hotel was clean and comfortable. We had a very spacious room. The hotel staff was wonderful and friendly. Stanley and Danielle you two rock!
3. Ughh. So I have called them with questions a few times and its sooo frustrating! I understand I have to keep getting redirected and being put on hold and etc, but some of the representatives I talk to are so rude! At one point, I was talking and the lady cut me off and said "hold…
4. This place is great in theory but the execution borders on offensive. I'm admittedly fresh from an interaction with front desk manager Ali, who --when I asked why my room wasn't made up despite my calling housekeeping, stopping by the front desk and activating the green "make up …
5. Having an awesome stay. Had to be somewhere very early so I parked in the valet and brought my bags into the lobby hoping they could hold them for me until check in. Turns out, they had a room available, and I was able to get right in around 6am. Truly grateful as I woke up at 4:…

### Block 6 — cluster “room / hotel / bed”
1. The hotel staff can use some help. They were friendly but not very professional. I did really enjoy the room though it was like staying in a studio apartment. The bed was definitely comfortable. I did enjoy the amenities the room had. The sink hat filtered water which was nice. T…
2. Beautiful hotel and facilities. Large king bedroom is quite spacious. Very clean, nice linens, nice bathroom. Friendly staff. Amazing light and music show. Close to several restaurants and venues. My only wish would've been a bathtub. Don't miss this gorgeous historic hotel.
3. We've stayed here several times and are always pleased. It is a beautiful hotel with the indoor gardens and water falls. However, be prepared for long walks especially if you get turned around which is easy to do here. The hotel is massive. The kids love it because of the "jungle…
4. Comfortable beds! The hotel is overall very good, but most impressive were the beds... they're comfortable! Most hotels cheap out & buy hard (uncomfortable) beds because they last longer, but Drury's were great and we slept really well. The location is good for the value, and the…
5. My sister in SB found this hotel purely by accident, and what a find! The deluxe King with balcony was comfy! The rooms & bathrooms are updated and clean. The sheets were luxurious. The front desk service is GREAT!!! Check in after 10p proved to be simple. While time in the room …

### Block 7 — cluster “city / philly / hotel”
1. We had an absolutely excellent stay at the Independent Hotel. It had adorable and unique rooms. Exposed brick, unique windows, and tasteful decor made the room extremely comfortable as well as chic. It was impeccably clean and felt crisp and airy. The staff was very friendly and …
2. This hotel is awesome!! Very close to Philly and tons of restaurants and shopping. The room was clean and comfortable. Got a great night sleep. The pillows and the bed was great. The hotel was quiet and the staff were extremely friendly and helpful. I would definitely stay here a…
3. Well worth the trip up to Reno from the Bay Area for a Mom's only weekend at the spa. So relaxing and entertaining. We rarely left the casino and spa. It's been remodeled since the last time I was there, so everything was fresh and new. Wish I could say the same for the rest of t…
4. I've been staying at this hotel for about four weeks (on work assignment in Trenton) and it's definitely one of the most modern hotels I've stayed in. The rooms are nicely appointed and the bed is amazingly comfortable. It's a really popular spot for business travelers and it's d…
5. Great hotel if you're staying in Philadelphia and want a hotel in the center of the action. This Doubletree is in city center and if you stay on one of the top floors, it's a great view of City Hall and the historic buildings of the city. It's also an easy walk to tons of restaur…

### Block 8 — cluster “great / restaurant / inn”
1. The SPA is my greatest new find (plan on spending an entire day here), they have many areas just to lounge, incredible services, great snacks to keep you nourished.. We stayed in the Tuscany Tower on a non-smoking floor and the room was fabulous.
2. We've been here three times and on each visit Djorn has taken excellent care of us. (I may have said "Better than family" once.) This was definitely a great break after lots of walking and the new grey robes feel AMAZING. Make sure to check out the saltwater pool and hot tub, too…
3. Everything about this place was wonderful. The waitress was lovely and I loved the architecture. The food was amazing! I had a salad and green beans which were probably one of the greatest things I've ever had before. I definitely recommend coming here. Everyone is so nice! They …
4. Did a weekend get away with my family here. Had a great time. All the folks we met were great with good food in the restaurants and making up of our room. No complaints.
5. Overall a fun place to be! Played some slots and had dinner at the cafe. There was live music but because of that you couldn't really talk at dinner. Food was mediocre though.

### Block 9 — cluster “great / rooms / stay”
1. The lovely ladies of Rathbone Mansions get a five star rating. We were here in early June, and we were pleasantly surprised. I only say surprised, because of some of the previous reviews. I completely agree with the previous reviewer who said if you want a Hilton, stay at a Hilto…
2. My friend recommended this place and I am glad she did! The room was very cute! We got the king room with hot tub and fireplace. Utilized both and had such a romantic weekend. It was pretty close to everything too. The only negative is the TV is pretty old but we barely watched T…
3. Good stay, comfortable beds, modern decor, everything I was looking for. It's across the street from a few fast food joints and has a big park lots.
4. Great location quiet and amazing room. Close to freeway and airport but can hardly hear it. Nice sitting room and huge HD tv.
5. Our stay here was fantastic. The rooms are very modern and spacious. The staff was very friendly in helping us with any questions. The rooms are very quiet and the beds are extremely comfortable.

### Block 10 — cluster “nashville / hotel / downtown”
1. Ooppps, didn't finish my review... Went to the Gaylord Opryland Resort while we were in Nashville. Did not stay there but came to visit the grounds and take the tour of the resort. After looking at the rates for the evening, the rates were very similar to those where we were stay…
2. Too big, too expensive and too far from Downtown. I was here for a conference, which seemed like an awesome idea. It wasn't. I'm in Direct Sales and (like most people at this conference) don't have tons of extra money to spend on mediocre meals when I'm traveling. $10 for a slice…
3. I absolutely loved this. We parked at opry mills near the movie theatre and just walked over. It was seriously impressive and worth all the hassle of the ridiculous traffic. So fancy inside and so many great photo ops! Definitely go on your visit to Nashville!
4. Great stay! Wonderful customer service, very comfortable beds and delicious breakfast buffet. The indoor "mall" is great if the weather gets bad. Many good restaurants in the area also.
5. My mother booked this hotel for a 1 night stay so we could take my 5 nieces and nephews to Nashville shores and this location was literally 5 minutes away. We had some issues with the owners who ran the place. They weren't the best or friendliest. And the morning of check out, th…

### Block 11 — cluster “noise / sleep / bed”
1. I had the fresh red fish and it was amazing. The atmosphere was great and our server was awesome. Make sure you get a window seat.
2. Pluses: 1. Location- short walk to Broadway and even shorter to Printer's Row. 2. Staff- everyone we encountered was just plain NICE! 3. Decor- room and lobby were great...bring flip flops, the floors are original stone which looks cool, but I just prefer carpet when I get out of…
3. Sat. Jun. 10,17 Room 115: Made a stop in SB late evening arrived at 8pm to leave at 7am. Next morning. The Place is pretty clean. Being that I had stayed here before about 5yrs ago. So I called to see if there were any rooms available, which there was, they quoted me a price but …
4. Three night stay. Not plush but not a total dump. East side was noisy from a factory air vent and cable was short on channels but right near airport and Pima air museum with several breakfast spots (Waffle House and Denny's) close.
5. The room was rundown and right next to a watercooler. There is literally no soundproofing so I had the pleasure of listening to teenage girls pound on different doors all night long. I had to call security to deal with the noise issue and it even took them quite a bit of time to …

### Block 12 — cluster “pool / water / got”
1. The king suite is great. The laundry was clean and well-kept. The pool was small but was able to get a few laps in. The gym was a winner it was packed every evening... The hot cinder they had in the lobby was a real treat and a great addition. The restaurant was great with a quai…
2. The food here is amazing and our server Geri is one of the best servers I have ever had. The char grilled oysters are a MUST. The reason I gave them 4 stars is because the manager was rude. Initially we asked if we could move our table so we can see the parade and he responded sa…
3. We stayed here for 3 nights late may. Great hotel. The rooms are clean and the staff are helpful. We stayed in the corner room which had a great view. The pool is on the roof. The water in the pool was very murky and very unappealing to swim In. And pool bar was awful. The roofto…
4. I've stayed in the old tower in a jacuzzi suite and in the new tower in a regular suite. They are excellent rooms with plenty of attention to detail. The rooms are comfortable and if you get the choice, stay on the Mt. Rose side (it's a great view to wake up to). Be careful with …
5. I like that they had water stations for the conference goers. I was able to stay hydrated.

### Block 13 — cluster “staff / nice / friendly”
1. From the moment we arrived the front desk staff was feiendly and welcomjng. They provided us with information on what to do close by the hotel and provided us recommendations amd tips for our hike at Sabino Canyon. The service throughout our stay was amazing. The Gringo Grill sta…
2. Rude lady working in the office. Lots of rules. Small spaces. Too expensive for what the park has to offer. The staff is always trying to get into people's business. Good location. Park is about average for cleanliness. Some RV spaces very cluttered. Probably would not stay again…
3. Nice and clean! The staff was friendly and made great suggestions on places we should try. Very nice rooms. Only complaint was they said they had a pool but didn't mention it was at the Marriott 2 blocks away.
4. Good stay for the road warrior. Nice staff and convenient for where I needed to be.
5. Star 1: Clean Comfortable room, and through the auspices of Kayak, one of the cheapest choices in town. Star 2: Safe surrounding neighborhood. The cheapest hotel choices are frequently in the dodgier areas of town... not in this case! Star 3: Climate control pt. 1: outstanding a/…

### Block 14 — cluster “breakfast / nice / room”
1. Nice stay hotel nicely decorated room OK nice comfortable bed after being out and about, paid $25 to park stated in/out but just used trolley to get around. Located about 4 blocks from Harrah's Casino and Bourbon Street. Breakfast was nice eggs oatmeal biscuits sausage patty, bre…
2. The staff here is amazing. I stayed here for two weeks with a big group for work, and management, front desk, servers, and drivers were friendly and accommodating. There is a clean and decently equipped gym with tvs; the laundry room was equipped with 4-5 washers and dryers and a…
3. It was good, not great. Good location. Receptionist was very helpful but the hotel didn't have a toothbrush in stock.
4. Beautiful lobby and helpful staff all the way through from check-in to check-out. There are plenty of cabs outside if you do not want to take mass transit. If you do need the train or bus, there is a subway stop one short block away and numerous SEPTA bus stops up and down Market…
5. Nice rooms and shower pressure was great. Check in was fast and staff is friendly. Free breakfast. Hotel is right near the on / off ramp of the interstate. So its very easy to get to.

### Block 15 — cluster “room / free / wifi”
1. There are so many wonderful things to say about this hotel: 1. Free Wi-Fi 2. Free Shuttle Downtown 3. Free Snacks 4. Cute Greeting (Post Card, Guitar Picks and Goo Goo Cluster) 5. Uber Helpful Staff (We had to leave a night early due to weather and they worked with Hotels.com to …
2. Once again, I wish that I could give half stars, because this is really a 4.5 review. Simply put, we were very happy with almost everything about this hotel. Its location cannot be beat -- just across the street from the French Quarter proper, on a major bus line, and about a 7-m…
3. Marriott could care less about helping you. Wow. Love this hotel. Hate the mgmt Also. It's music city USA and u can't hardly hear music in the hotel
4. Was worried with bad reviews but I had a comfortable stay on 5th floor. Free wifi worked. Bed and linens a bit rough, but pillows were comfy. One free bottled water in room, mini fridge in room. Not much of a closet had in room safe and coffee service. Shower only, but had good b…
5. It's a new boutique hotel located out in the middle of nowhere so both the wireless and wired internet don't work for business travelers located especially on the fourth floor. Management is helpless to do anything to restore the internet connection so they will just give you the…

### Block 16 — cluster “marriott / room / service”
1. We paid $230 per night for our stay during the Memorial Weekend and man how I wish I stayed at Motel 6 at next door instead. The location is prime - walkable to tons of cute little restaurant and even Whole Foods. However, the room was way too shabby - we can hear very loud TV no…
2. Nice hotel and gorgeous views with the plants, lights and fountains. It's a Marriott property but without the Marriott customer service. $25.00 for a long line breakfast buffet for mediocre food. Parking is insane...even valet is a long wait. The ability to acquire tickets to eve…
3. Very nice high end Marriott Hotel. It is pricey but worth it. Check out the deals on their website, we got a Labor Day special that included, breakfast, and parking for $199, Both side of the hotel have great views. If you are a Gold Marriott or higher member, you get additional …
4. The TV shows are $4.99 and they have commercials! What a cheesy way to make money and a sign of a less than classy hotel, particularly when you pay more than $150 a night. And there is NO complimentary breakfast, just an overpriced buffet, something even the cheapest hotels in Ca…
5. Nice rooms and shower pressure was great. Check in was fast and staff is friendly. Free breakfast. Hotel is right near the on / off ramp of the interstate. So its very easy to get to.

### Block 17 — cluster “tampa / hyatt / grand”
1. Stayed here for one night prior to cruising out. Having access to the shuttle to pick my group up from the airport, take us to a restaurant for dinner and Walmart was absolutely great. Richard the shuttle driver kept us in stitches. The room was clean and neat. The breakfast was …
2. The Grand Hyatt in Tampa Bay is one of the best resort experiences my wife and i have had in awhile. We visited Tampa Bay recently and due to the reviews on multiple websites we decided to stay at the Grand Hyatt. Here is a rundown of my experience: - Staff: The staff is hands do…
3. Decent hotel in downtown Tampa. Stayed in the suite with the fold out couch. This place has a happy hour with FREE drinks, which we missed, but how cool. The complimentary breakfast was awesome. Made to order omelets, fresh fruit, cereal, toast, bacon, sausage, home fries, juice …
4. My Aunt and I stayed here for a few nights in December. The location is GREAT being very close to State St., pier, zoo etc.. The staff was AMAZING!! The girls who work in the front were extremely helpful, even offering us local family owned options for dining. The chef, Jenny doe…
5. I stayed here with some friends after a concert in town and the Hyatt always delivers with a stellar reputation for class. I was impressed and liked the room although I was just visiting here. I live in St. Louis so I would not need a hotel otherwise, but I can tell travelers thi…

### Block 18 — cluster “beach / hotel / palm”
1. Service great, upstairs bar great, pool area nice. Bed mediocre, pillows mediocre, no on demand or movie rentals available, only one elevator working so lots of waiting. Super close to beach, and nice views.
2. Wonderful place to stay & enjoy some beach time! We stayed in room 202 & it was nice, spacious, modernized, & comfortable! The place in General has plenty of parking, security, private pool, private elevator, & a wonderful BBQ area with tables & chairs & most importantly in front…
3. Had a great experience staying at the Pier House. Hotel staff were friendly, helpful and awesome. Stayed on the 10th floor and heard no noise from the bar. Views of the beach in the morning are gorgeous. Nothing came up missing from my personal belongings at all and guest service…
4. Holiday Inn Express is a good place to stay for a night while on a business trip. Did I miss having room service, yes. Overall, I would recommend this location.
5. When I was very little I grew up on Clearwater Beach, living a short distance from this hotel, then known (I think) as the Gulf View Apartments. I would have figured this long-time beach fixture would have been demolished long ago to build some other high-rise hotel or condo, but…

### Block 19 — cluster “wedding / anniversary / linda”
1. I fell in love with the place the moment I walked in! My (now) wife and I decided to have our wedding here because of the beautiful location, the wonderfully friendly proprietors, and the convenience of being able to have the lodging, ceremony, and reception all in the same place…
2. We recently spent our 19th wedding anniversary at the rebrand Fess Parker/Red Lion/ Doubletree hotel. It was a staycation and we were only there for 1 night. We'd stayed here before when it was a doubletree so we knew what to expect. When they learned it was are in a versary they…
3. bad room,small bed, so noise I heard from 2nd Floor. I spent almost a hour to find my room from check in
4. Tenisha was lovely. I was in a very cross mood with my husband and the key failed to work. Had to go back to the desk and she was very kind and apologetic, offered free snacks for which we declined. The room is exquisite. The linens and pillows made me feel like I was sleeping in…
5. hulamom and i stayed here during her birthday jaunt and it was actually pretty lovely. the location is pretty good, especially if you're a newbie and you don't ever want to leave the quarter. commander's palace (hulamom's fave) was booked her b'day night but the concierge got us …

### Block 20 — cluster “staff / breakfast / room”
1. Great! 100% of the staff were engaging and helpful. I prepaid for a King room and the bellman escorted us to a double queen in a new portion of the hotel. I called the desk clerk and he apologized and had the bellman take us to the historic side of the hotel and put us in a top n…
2. My husband and I stayed for one night. Upon arrival, the check in was easy and fast but the front desk associate could have been a little more professional in the way she looked and handled our reservation. She did not offer any direction as to how to get to our room. She was alr…
3. But, It might be a 2 stars. All of the staff were great, but the 2 room suite with a small version of a kitchen was more like a 2. Poorly done "remodel" or just plain in need of remodeling. Bummer
4. Staff is amazing here. So helpful and supportive when we stayed in 2015. They knew we had a lot of stuff to load in our car, plus two little kids, so they let us take our time. We were way past the checkout hour by the time we left, lol! Hotel room was always clean, house keeping…
5. Overall, this is a nicely appointed hotel with nice rooms, good service, and good amenities. The exterior and facade is beautiful as is the common space in the hotel. Carousel bar is nice and unique--definitely best hotel bar in NOLA (loews swizzle stick i like too), but is often…

---

<details><summary>Answer key</summary>

- Block 0: intruder is #5 (from cluster 13, “staff / nice / friendly”)
- Block 1: intruder is #3 (from cluster 7, “city / philly / hotel”)
- Block 2: intruder is #5 (from cluster 17, “tampa / hyatt / grand”)
- Block 3: intruder is #5 (from cluster 15, “room / free / wifi”)
- Block 4: intruder is #3 (from cluster 19, “wedding / anniversary / linda”)
- Block 5: intruder is #2 (from cluster 6, “room / hotel / bed”)
- Block 6: intruder is #3 (from cluster 8, “great / restaurant / inn”)
- Block 7: intruder is #3 (from cluster 0, “casino / reno / peppermill”)
- Block 8: intruder is #1 (from cluster 0, “casino / reno / peppermill”)
- Block 9: intruder is #4 (from cluster 11, “noise / sleep / bed”)
- Block 10: intruder is #4 (from cluster 8, “great / restaurant / inn”)
- Block 11: intruder is #1 (from cluster 2, “oysters / good / shrimp”)
- Block 12: intruder is #2 (from cluster 2, “oysters / good / shrimp”)
- Block 13: intruder is #5 (from cluster 14, “breakfast / nice / room”)
- Block 14: intruder is #3 (from cluster 6, “room / hotel / bed”)
- Block 15: intruder is #3 (from cluster 16, “marriott / room / service”)
- Block 16: intruder is #5 (from cluster 14, “breakfast / nice / room”)
- Block 17: intruder is #4 (from cluster 8, “great / restaurant / inn”)
- Block 18: intruder is #4 (from cluster 3, “hotel / great / nice”)
- Block 19: intruder is #3 (from cluster 11, “noise / sleep / bed”)
- Block 20: intruder is #5 (from cluster 1, “bar / orleans / new”)

</details>

---

## Decision (to be completed by a human reviewer)

- [ ] I read the inspection sheets above for every finalist.
- [ ] I took the intruder tests; clusters whose intruder was not obvious are noted below.
- [ ] Winner: `____________` — because: ____________
- [ ] The sentence below is true and may be quoted in the final recommendation:

> a human reviewed the clusters of the winning pipeline and confirmed they are thematically coherent

Record the confirmation in the HITL app (`streamlit run src/reviewscope_ml/hitl/app.py`), which persists it to `data/feedback/` with reviewer name and timestamp.


In [5]:
# Side-by-side 2-D scatter of the four runs
import matplotlib.pyplot as plt
from reviewscope_ml.pipelines import load_run
from reviewscope_ml.pipelines.spec import VARIANTS

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, variant in zip(axes, VARIANTS):
    art = load_run(cfg.runs_dir / f'{variant}__{cfg.sample_size}__s42')
    noise = art.labels == -1
    ax.scatter(art.coords_2d[noise, 0], art.coords_2d[noise, 1], s=2, c='lightgrey', alpha=0.3)
    ax.scatter(art.coords_2d[~noise, 0], art.coords_2d[~noise, 1], s=2,
               c=art.labels[~noise], cmap='tab20', alpha=0.6)
    ax.set_title(f"{variant}\n{art.metrics['n_clusters']} clusters, "
                 f"noise {art.metrics['noise_ratio']:.0%}")
    ax.axis('off')
plt.tight_layout(); plt.show()

## Next step — the human part

1. Read the **qualitative inspection** and take the **intruder test** in the
   report above for each shortlisted finalist.
2. Open the HITL app and record your verdict:
   ```bash
   streamlit run src/reviewscope_ml/hitl/app.py
   ```
3. Only after the sign-off is recorded does the comparison have a winner.
   See `11_hitl_roundtrip.ipynb` for how recorded feedback changes the next run.